In [1]:
import json

with open("../data/match.json") as f:
    data = json.load(f)

data

[{'Home team': 'Man United',
  'Away team': 'Brighton',
  'Final score': '4-2',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '24th',
    'player': 'Matheus Cunha',
    'team': 'Man United',
    'evidence': 'Cunha kept his cool after some battling play, taking a touch on the edge of the box and sending a precise right-footed effort beyond Bart Verbruggen.'},
   {'minute': '34th',
    'player': 'Casemiro',
    'team': 'Man United',
    'evidence': 'Casemiro, whose long-range strike hit Yasin Ayari and wrongfooted Verbruggen.'},
   {'minute': '61st',
    'player': 'Bryan Mbeumo',
    'team': 'Man United',
    'evidence': "Mbeumo, who squeezed a low shot through Dunk's legs and past Verbruggen at his near post in front of the Stretford End in the 61st minute."},
   {'minute': '74th',
    'player': 'Danny Welbeck',
    'team': 'Brighton',
    'evidence': "Welbeck's free-kick and teenage substitute Charalampos Kostoulas' stoppage-time header had the hosts wobbling."},
   {'minute': '90

In [5]:
# Code that produces first Excel spreadsheet (match level data)

import pandas as pd

match_rows = []
for match in data:
    match_rows.append({
        "home team": match["Home team"],
        "home manager": match["home_manager"],
        "home score": match["Final score"].split("-")[0],
        "away team": match["Away team"],
        "away manager": match["away_manager"],
        "away score": match["Final score"].split("-")[1],
        "stadium": match["Ground"]
    })
matches_df = pd.DataFrame(match_rows)
matches_df

,home team,home manager,home score,away team,away manager,away score,stadium
0,Man United,Ruben Amorim,4,Brighton,Fabian Hurzeler,2,Old Trafford
1,Man United,Ruben Amorim,4,Bournemouth,Andoni Iraola,4,Old Trafford


In [6]:
def summarise_goals(goals):
    return ", ".join([
        f"{g['player']} ({g['minute']})"
        for g in goals
    ])

matches_df["goals_summary"] = [
    summarise_goals(match.get("goals", []))
    for match in data
]

matches_df

,home team,home manager,home score,away team,away manager,away score,stadium,goals_summary
0,Man United,Ruben Amorim,4,Brighton,Fabian Hurzeler,2,Old Trafford,"Matheus Cunha (24th), Casemiro (34th), Bryan M..."
1,Man United,Ruben Amorim,4,Bournemouth,Andoni Iraola,4,Old Trafford,"Amad Diallo (13), Casemiro (40), Bruno Fernand..."


In [7]:
from collections import Counter

goal_scorers = []
for match in data:
    for goal in match.get("goals", []):
        goal_scorers.append(goal["player"])

counts = Counter(goal_scorers)

ranking_df = (
    pd.DataFrame(
        counts.items(),
        columns=["Player", "Goals Seen Live"]
    )
    .sort_values(
        ["Goals Seen Live", "Player"],
        ascending=[False, True]
    )
)

ranking_df["Rank"] = (
    ranking_df["Goals Seen Live"]
    .rank(method="min", ascending=False)
    .astype(int)
)

ranking_df = ranking_df[
    ["Rank", "Player", "Goals Seen Live"]
].reset_index(drop=True)

ranking_df

,Rank,Player,Goals Seen Live
0,1,Bryan Mbeumo,2
1,1,Casemiro,2
2,1,Matheus Cunha,2
3,4,Amad Diallo,1
4,4,Bruno Fernandes,1
5,4,Charalampos Kostoulas,1
6,4,Danny Welbeck,1
7,4,Eli Junior Kroupi,1
8,4,Evanilson,1
9,4,Marcus Tavernier,1


In [8]:
with pd.ExcelWriter("../data/enriched_matches.xlsx", engine="openpyxl") as writer:
    matches_df.to_excel(writer, sheet_name="Matches", index=False)
    ranking_df.to_excel(writer, sheet_name="Top Scorers", index=False)